In [ ]:
# Dynamic BSE 100 Live Stock Analysis Tool
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import requests
from bs4 import BeautifulSoup
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [ ]:
class BSELiveDataFetcher:
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        })
        self.bse_100_stocks = []
    
    def fetch_bse_100_list(self):
        """Dynamically fetch current BSE 100 stock list from multiple sources"""
        print("🔄 Fetching current BSE 100 stock list...")
        
        # Method 1: Try BSE official website
        stocks = self._fetch_from_bse_official()
        if stocks:
            self.bse_100_stocks = stocks
            return stocks
        
        # Method 2: Try NSE website (many BSE stocks are also on NSE)
        stocks = self._fetch_from_nse()
        if stocks:
            self.bse_100_stocks = stocks
            return stocks
        
        # Method 3: Try financial data websites
        stocks = self._fetch_from_financial_sites()
        if stocks:
            self.bse_100_stocks = stocks
            return stocks
        
        # Fallback: Use a curated list of major BSE stocks
        print("⚠️ Using fallback stock list")
        return self._get_fallback_list()
    
    def _fetch_from_bse_official(self):
        """Fetch from BSE official API/website"""
        try:
            # BSE API endpoint for index constituents
            url = "https://api.bseindia.com/BseIndiaAPI/api/GetMktData/w"
            params = {
                'ordcol': 'TT',
                'strType': 'index',
                'strfilter': 'S&P BSE 100'
            }
            
            response = self.session.get(url, params=params, timeout=10)
            if response.status_code == 200:
                data = response.json()
                if 'Table' in data and data['Table']:
                    stocks = []
                    for item in data['Table']:
                        if 'scrip_cd' in item:
                            symbol = item['scrip_cd']
                            stocks.append(f"{symbol}.BO")
                    
                    if len(stocks) > 50:  # Reasonable check
                        print(f"✅ Fetched {len(stocks)} stocks from BSE official")
                        return stocks
        except Exception as e:
            print(f"❌ BSE official fetch failed: {str(e)[:50]}...")
        return None
    
    def _fetch_from_nse(self):
        """Fetch from NSE website (many stocks are common)"""
        try:
            # NSE API for NIFTY 100 (similar to BSE 100)
            url = "https://www.nseindia.com/api/equity-stockIndices?index=NIFTY%20100"
            
            response = self.session.get(url, timeout=10)
            if response.status_code == 200:
                data = response.json()
                if 'data' in data:
                    stocks = []
                    for item in data['data']:
                        if 'symbol' in item:
                            symbol = item['symbol']
                            # Convert NSE symbols to BSE format
                            stocks.append(f"{symbol}.BO")
                    
                    if len(stocks) > 50:
                        print(f"✅ Fetched {len(stocks)} stocks from NSE (converted to BSE)")
                        return stocks
        except Exception as e:
            print(f"❌ NSE fetch failed: {str(e)[:50]}...")
        return None
    
    def _fetch_from_financial_sites(self):
        """Fetch from financial data websites"""
        try:
            # Try screener.in or similar sites
            url = "https://www.screener.in/api/company/search/?q=BSE"
            
            response = self.session.get(url, timeout=10)
            if response.status_code == 200:
                # Parse response and extract stock symbols
                # This would need to be customized based on the actual API response
                pass
        except Exception as e:
            print(f"❌ Financial sites fetch failed: {str(e)[:50]}...")
        return None
    
    def _get_fallback_list(self):
        """Fallback list of major BSE stocks"""
        return [
            'RELIANCE.BO', 'TCS.BO', 'HDFCBANK.BO', 'INFY.BO', 'HINDUNILVR.BO',
            'ICICIBANK.BO', 'KOTAKBANK.BO', 'SBIN.BO', 'BHARTIARTL.BO', 'ITC.BO',
            'ASIANPAINT.BO', 'LT.BO', 'AXISBANK.BO', 'MARUTI.BO', 'SUNPHARMA.BO',
            'TITAN.BO', 'ULTRACEMCO.BO', 'WIPRO.BO', 'NESTLEIND.BO', 'POWERGRID.BO',
            'NTPC.BO', 'BAJFINANCE.BO', 'HCLTECH.BO', 'ONGC.BO', 'TATAMOTORS.BO',
            'TECHM.BO', 'COALINDIA.BO', 'ADANIPORTS.BO', 'TATASTEEL.BO', 'BAJAJFINSV.BO',
            'GRASIM.BO', 'HINDALCO.BO', 'DRREDDY.BO', 'CIPLA.BO', 'JSWSTEEL.BO',
            'BRITANNIA.BO', 'INDUSINDBK.BO', 'EICHERMOT.BO', 'HEROMOTOCO.BO', 'UPL.BO',
            'BAJAJ-AUTO.BO', 'DIVISLAB.BO', 'APOLLOHOSP.BO', 'TATACONSUM.BO', 'GODREJCP.BO',
            'PIDILITIND.BO', 'DABUR.BO', 'MARICO.BO', 'COLPAL.BO', 'BERGEPAINT.BO',
            'HAVELLS.BO', 'VOLTAS.BO', 'WHIRLPOOL.BO', 'PAGEIND.BO', 'MCDOWELL-N.BO',
            'SIEMENS.BO', 'BOSCHLTD.BO', 'ABB.BO', 'HONAUT.BO', 'CUMMINSIND.BO',
            'TORNTPHARM.BO', 'BIOCON.BO', 'CADILAHC.BO', 'LUPIN.BO', 'AUROPHARMA.BO',
            'GAIL.BO', 'IOC.BO', 'BPCL.BO', 'HINDPETRO.BO', 'PETRONET.BO',
            'VEDL.BO', 'SAIL.BO', 'NMDC.BO', 'MOIL.BO', 'NATIONALUM.BO',
            'BANKBARODA.BO', 'PNB.BO', 'CANBK.BO', 'UNIONBANK.BO', 'IDFCFIRSTB.BO',
            'FEDERALBNK.BO', 'RBLBANK.BO', 'BANDHANBNK.BO', 'AUBANK.BO', 'YESBANK.BO',
            'LICHSGFIN.BO', 'SRTRANSFIN.BO', 'M&MFIN.BO', 'BAJAJHLDNG.BO', 'SHREECEM.BO',
            'AMBUJACEM.BO', 'ACC.BO', 'RAMCOCEM.BO', 'JKCEMENT.BO', 'HEIDELBERG.BO',
            'TRENT.BO', 'ADITYADB1.BO', 'JUBLFOOD.BO', 'WESTLIFE.BO', 'ZOMATO.BO'
        ]
    
    def get_live_market_data(self, symbol):
        """Get live market data for a single stock"""
        try:
            # Method 1: Yahoo Finance (most reliable)
            ticker = yf.Ticker(symbol)
            info = ticker.info
            hist = ticker.history(period='5d')
            
            if not hist.empty:
                current_price = hist['Close'].iloc[-1]
                prev_close = hist['Close'].iloc[-2] if len(hist) > 1 else current_price
                change = current_price - prev_close
                change_pct = (change / prev_close) * 100 if prev_close != 0 else 0
                
                return {
                    'symbol': symbol,
                    'current_price': current_price,
                    'previous_close': prev_close,
                    'change': change,
                    'change_percent': change_pct,
                    'volume': hist['Volume'].iloc[-1] if 'Volume' in hist.columns else 0,
                    'market_cap': info.get('marketCap', 0),
                    'pe_ratio': info.get('trailingPE', None),
                    'sector': info.get('sector', 'Unknown'),
                    'last_updated': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                }
        except Exception as e:
            print(f"Error fetching live data for {symbol}: {str(e)[:50]}...")
        return None
    
    def get_live_market_summary(self):
        """Get live BSE market summary"""
        try:
            # Fetch BSE SENSEX data
            sensex = yf.Ticker('^BSESN')
            sensex_data = sensex.history(period='2d')
            
            if not sensex_data.empty:
                current = sensex_data['Close'].iloc[-1]
                prev = sensex_data['Close'].iloc[-2] if len(sensex_data) > 1 else current
                change = current - prev
                change_pct = (change / prev) * 100 if prev != 0 else 0
                
                return {
                    'sensex_level': current,
                    'sensex_change': change,
                    'sensex_change_pct': change_pct,
                    'last_updated': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                }
        except Exception as e:
            print(f"Error fetching market summary: {e}")
        return None

In [ ]:
class LiveBSEAnalyzer:
    def __init__(self):
        self.data_fetcher = BSELiveDataFetcher()
        self.live_data = {}
        self.historical_data = {}
    
    def initialize(self):
        """Initialize with dynamic stock list"""
        self.stock_list = self.data_fetcher.fetch_bse_100_list()
        print(f"📊 Initialized with {len(self.stock_list)} stocks")
        return len(self.stock_list)
    
    def fetch_live_data_batch(self, max_workers=15):
        """Fetch live data for all stocks concurrently"""
        print("🔄 Fetching live market data...")
        successful = 0
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_stock = {
                executor.submit(self.data_fetcher.get_live_market_data, stock): stock 
                for stock in self.stock_list
            }
            
            for future in as_completed(future_to_stock):
                stock = future_to_stock[future]
                try:
                    result = future.result()
                    if result:
                        self.live_data[stock] = result
                        successful += 1
                        if successful % 20 == 0:
                            print(f"✅ Loaded {successful} live quotes...")
                except Exception as e:
                    print(f"❌ Failed {stock}: {str(e)[:30]}...")
        
        print(f"\n📈 Live data loaded for {successful} stocks")
        return successful
    
    def get_market_movers(self):
        """Get top gainers and losers"""
        if not self.live_data:
            return None, None
        
        df = pd.DataFrame(self.live_data.values())
        
        # Top gainers
        top_gainers = df.nlargest(10, 'change_percent')[[
            'symbol', 'current_price', 'change', 'change_percent', 'volume', 'sector'
        ]]
        
        # Top losers
        top_losers = df.nsmallest(10, 'change_percent')[[
            'symbol', 'current_price', 'change', 'change_percent', 'volume', 'sector'
        ]]
        
        return top_gainers, top_losers
    
    def get_volume_leaders(self):
        """Get stocks with highest trading volume"""
        if not self.live_data:
            return None
        
        df = pd.DataFrame(self.live_data.values())
        volume_leaders = df.nlargest(15, 'volume')[[
            'symbol', 'current_price', 'change_percent', 'volume', 'sector'
        ]]
        
        return volume_leaders
    
    def get_sector_performance(self):
        """Get sector-wise performance"""
        if not self.live_data:
            return None
        
        df = pd.DataFrame(self.live_data.values())
        sector_perf = df.groupby('sector').agg({
            'change_percent': ['mean', 'count'],
            'current_price': 'mean',
            'volume': 'sum'
        }).round(2)
        
        sector_perf.columns = ['Avg_Change_%', 'Stock_Count', 'Avg_Price', 'Total_Volume']
        return sector_perf.sort_values('Avg_Change_%', ascending=False)
    
    def generate_live_report(self):
        """Generate comprehensive live market report"""
        print("\n" + "="*60)
        print("📊 LIVE BSE MARKET REPORT")
        print("="*60)
        
        # Market summary
        market_summary = self.data_fetcher.get_live_market_summary()
        if market_summary:
            print(f"\n🏛️ SENSEX: {market_summary['sensex_level']:.2f} ")
            print(f"   Change: {market_summary['sensex_change']:+.2f} ({market_summary['sensex_change_pct']:+.2f}%)")
            print(f"   Updated: {market_summary['last_updated']}")
        
        # Market movers
        gainers, losers = self.get_market_movers()
        
        if gainers is not None:
            print("\n🚀 TOP GAINERS:")
            gainers['symbol'] = gainers['symbol'].str.replace('.BO', '')
            print(gainers[['symbol', 'current_price', 'change_percent', 'sector']].round(2).to_string(index=False))
        
        if losers is not None:
            print("\n📉 TOP LOSERS:")
            losers['symbol'] = losers['symbol'].str.replace('.BO', '')
            print(losers[['symbol', 'current_price', 'change_percent', 'sector']].round(2).to_string(index=False))
        
        # Volume leaders
        volume_leaders = self.get_volume_leaders()
        if volume_leaders is not None:
            print("\n📊 VOLUME LEADERS:")
            volume_leaders['symbol'] = volume_leaders['symbol'].str.replace('.BO', '')
            print(volume_leaders[['symbol', 'current_price', 'change_percent', 'volume']].round(2).to_string(index=False))
        
        # Sector performance
        sector_perf = self.get_sector_performance()
        if sector_perf is not None:
            print("\n🏭 SECTOR PERFORMANCE:")
            print(sector_perf.to_string())
        
        print("\n" + "="*60)
        print(f"Report generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("="*60)
    
    def export_live_data(self):
        """Export live data to CSV"""
        if not self.live_data:
            print("❌ No live data to export")
            return
        
        df = pd.DataFrame(self.live_data.values())
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"bse_live_data_{timestamp}.csv"
        df.to_csv(filename, index=False)
        print(f"💾 Live data exported to: {filename}")

In [ ]:
# Initialize and run live BSE analysis
print("🚀 Starting Live BSE Analysis...")
analyzer = LiveBSEAnalyzer()

# Step 1: Get dynamic stock list
stock_count = analyzer.initialize()

if stock_count > 0:
    # Step 2: Fetch live data
    start_time = time.time()
    live_count = analyzer.fetch_live_data_batch()
    end_time = time.time()
    
    print(f"\n⏱️ Live data fetched in {end_time - start_time:.1f} seconds")
    
    if live_count > 0:
        # Step 3: Generate comprehensive report
        analyzer.generate_live_report()
        
        # Step 4: Export data
        analyzer.export_live_data()
    else:
        print("❌ No live data available")
else:
    print("❌ Failed to fetch stock list")

In [ ]:
# Quick refresh function for live updates
def refresh_live_data():
    """Quick function to refresh live data"""
    print("🔄 Refreshing live data...")
    analyzer.fetch_live_data_batch()
    analyzer.generate_live_report()

# Uncomment to refresh data
# refresh_live_data()

In [ ]:
# Create custom watchlist from BSE stocks
def create_custom_watchlist(symbols):
    """Create a custom watchlist with specific stocks"""
    print(f"📋 Creating watchlist for {len(symbols)} stocks...")
    
    watchlist_data = {}
    for symbol in symbols:
        if not symbol.endswith('.BO'):
            symbol = f"{symbol}.BO"
        
        data = analyzer.data_fetcher.get_live_market_data(symbol)
        if data:
            watchlist_data[symbol] = data
    
    if watchlist_data:
        df = pd.DataFrame(watchlist_data.values())
        df['symbol'] = df['symbol'].str.replace('.BO', '')
        
        print("\n📊 CUSTOM WATCHLIST:")
        display_cols = ['symbol', 'current_price', 'change', 'change_percent', 'volume', 'sector']
        print(df[display_cols].round(2).to_string(index=False))
        
        return df
    return None

# Example: Create watchlist for specific stocks
my_watchlist = ['RELIANCE', 'TCS', 'HDFCBANK', 'INFY', 'ICICIBANK']
watchlist_df = create_custom_watchlist(my_watchlist)